In [0]:
# MAGIC %md
# MAGIC # 02 - Gold: Visão por UF
# MAGIC
# MAGIC Fonte: gold.fato_alfabetizacao_municipio
# MAGIC Saída: gold.visao_uf (Delta, partitionBy ano, ZORDER)
# MAGIC Granularidade: (ano, estado_sigla, rede)
# MAGIC Padrão: rastreabilidade + DQ + CTAS (compatível com serverless)

# COMMAND ----------

import sys
from pathlib import Path

repo = Path.cwd()
while not repo.name.startswith("postech-aisc") and repo.parent != repo:
    repo = repo.parent
sys.path.insert(0, str(repo))

from pyspark.sql import functions as F

# COMMAND ----------

# 1. LER O FATO GOLD
df_fato = spark.table("gold.fato_alfabetizacao_municipio")

print(f"[INFO] gold.fato_alfabetizacao_municipio: {df_fato.count():,} linhas")

# COMMAND ----------

# 2. AGREGAR POR (ano, estado_sigla, rede)
df_visao = (
    df_fato
    .groupBy("ano", "estado_sigla", "rede", "rede_nome")
    .agg(
        F.count("*").alias("total_municipios"),
        F.round(F.avg("resultado"), 2).alias("taxa_media"),
        F.round(F.avg("meta"), 2).alias("meta_media"),
        F.round(F.avg("folga_pp"), 2).alias("folga_media_pp"),
        F.sum(F.when(F.col("status_meta") == "ATINGIU", 1).otherwise(0)).alias("municipios_atingiram"),
        F.sum(F.when(F.col("status_meta") == "SEM_META", 1).otherwise(0)).alias("municipios_sem_meta"),
    )
    .withColumn(
        "pct_atingiram_meta",
        F.round(F.col("municipios_atingiram") / F.col("total_municipios") * 100, 2),
    )
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit("gold.fato_alfabetizacao_municipio"))
    .withColumn("version", F.lit("1.0"))
)

# COMMAND ----------

# 3. DATA QUALITY
chaves = ["ano", "estado_sigla", "rede"]

total = df_visao.count()
nulos_chave = df_visao.filter(
    F.expr(" OR ".join(f"{c} IS NULL" for c in chaves))
).count()
dups = df_visao.groupBy(*chaves).count().filter(F.col("count") > 1).count()

print(f"[DQ] Total de linhas: {total}")
print(f"[DQ] Nulos na chave composta: {nulos_chave}")
print(f"[DQ] Duplicados na chave composta: {dups}")

spark.sql("CREATE DATABASE IF NOT EXISTS monitoring")
spark.sql("""
  CREATE TABLE IF NOT EXISTS monitoring.dq_results (
    table_name STRING, rule STRING, status STRING,
    records_checked BIGINT, failures BIGINT, run_at TIMESTAMP
  ) USING DELTA
""")

registros = [
    ("gold.visao_uf", "completude_chave",
     "PASS" if nulos_chave == 0 else "FAIL", int(total), int(nulos_chave)),
    ("gold.visao_uf", "unicidade_chave_composta",
     "PASS" if dups == 0 else "FAIL", int(total), int(dups)),
]
df_dq = spark.createDataFrame(
    registros, ["table_name", "rule", "status", "records_checked", "failures"]
).withColumn("run_at", F.current_timestamp())

df_dq.createOrReplaceTempView("vw_dq_uf")
spark.sql("INSERT INTO monitoring.dq_results SELECT * FROM vw_dq_uf")

# COMMAND ----------

# 4. GRAVAR EM DELTA (CTAS, compatível com serverless)
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

df_visao.createOrReplaceTempView("vw_visao_uf")

spark.sql("""
CREATE OR REPLACE TABLE gold.visao_uf
USING DELTA
PARTITIONED BY (ano)
AS SELECT * FROM vw_visao_uf
""")

print(f"\n[OK] gold.visao_uf gravada | Registros: {spark.table('gold.visao_uf').count():,}")

spark.sql("OPTIMIZE gold.visao_uf ZORDER BY (estado_sigla, rede)")
print("[OK] ZORDER aplicado em (estado_sigla, rede)")

# COMMAND ----------

# 5. VERIFICAÇÃO RÁPIDA
spark.sql("""
  SELECT ano, estado_sigla, rede_nome,
         total_municipios, taxa_media, meta_media, pct_atingiram_meta
  FROM gold.visao_uf
  ORDER BY ano, estado_sigla, rede_nome
  LIMIT 20
""").show()

In [0]:
# Taxa por estado no fato
spark.sql("""
SELECT estado_sigla,
       COUNT(*) AS qtd,
       COUNT(resultado) AS com_taxa,
       ROUND(AVG(resultado), 2) AS taxa_media
FROM gold.fato_alfabetizacao_municipio
GROUP BY estado_sigla
ORDER BY estado_sigla
""").show()

# E o que há de NULL
spark.sql("""
SELECT estado_sigla, COUNT(*) AS nulos_taxa
FROM gold.fato_alfabetizacao_municipio
WHERE resultado IS NULL
GROUP BY estado_sigla
ORDER BY estado_sigla
""").show()